# Step 4: Upload subjects from RRN to Zooniverse using Panoptes
This step of the RRN-to-Zooniverse pipeline connects to Zooniverse, creates a new Zooniverse project and subject set, and uploads subject images and their associated metadata (manifest) to the project. 

In [1]:
import pandas as pd
import numpy as np

from astropy.table import Table as t

from panoptes_client import Panoptes, Workflow, Project, Subject, SubjectSet
import getpass

import functools
import os
import threading
from multiprocessing.pool import ThreadPool

from tqdm import tqdm
from tqdm import tqdm_notebook
from tqdm import trange


In [4]:
# Creating a system that allows me to check what's been uploaded

def update_txt_file(id_field):
    """Appends a new id_field to the uploaded_subjects_by_id_field.txt file."""
    with open(filename, 'a') as f:
        f.write(str(id_field) + '\n')

def get_uploaded_ids(filename):
    with open(filename, 'r') as f:
        uploaded_ids = [line.strip() for line in f]
    return np.array(uploaded_ids)

def check_if_uploaded(id_field):
    """Checks if the given id_field is already in uploaded_subjects_by_id_field.txt."""
    try:
        with open(filename, 'r') as f:
            uploaded_ids = {line.strip() for line in f}
        return str(id_field) in uploaded_ids
    except FileNotFoundError:
        return False  # If the file doesn't exist yet, nothing has been uploaded



def upload_selected_galaxies(selected_galaxies, project_id, subject_set_id, subject_png_save_dir, manifest_columns, progress_file, n_processes=10, batch_size=10):
    """
    Upload selected galaxies to Galaxy Zoo using threading (batching adds to subject set).

    Parameters:
    ----------
    selected_galaxies: Pandas DataFrame
        A Pandas DataFrame containing the galaxies selected for upload. Must include PNG file locations and metadata to be uploaded
    project_id: str
        Zooniverse project ID
    subject_set_id: str
        Zooniverse subject set ID tied to the project
    subject_png_save_dir: str
        directory on the RRN where subject PNG images are stored
    manifest_columns: list
        list of metadata columns to include in the manifest upload
    progress_file: str
        file for keeping track of which source ids have already been uploaded via Panoptes
    n_processes: int
        number of workers to launch for batch processing
    batch_size: int
        number of selected galaxies to upload at once
    """
    # Connect once globally
    client = Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))
    
    project = Project.find(project_id)
    subject_set = SubjectSet.find(subject_set_id)

    # Prepare manifest
    manifest = []
    uploaded_ids = get_uploaded_ids(progress_file) #keeps track of which src ids have already been uploaded via Panoptes
    print('uploaded_ids', uploaded_ids)

    for i in range(len(selected_galaxies)):
        row = selected_galaxies.loc[i]
        id_str = str(row['#id_str'])

        if id_str in uploaded_ids:
            continue

        entry = {
            'png_locs': [
                subject_png_save_dir + str(row['image1_path']),
                subject_png_save_dir + str(row['image2_path'])
            ],
            'metadata': {col: str(row[col]) for col in manifest_columns},
            'id_str': id_str
        }
        print('entry', entry)
        manifest.append(entry)

    print('Manifest created')
    pbar = tqdm(total=len(manifest), unit=' subjects uploaded')
    pbar_lock = threading.Lock() #keeps multiple threads from updated the progress bar at once (which could corrupt the display)

    # Split manifest into batches
    for batch_start in range(0, len(manifest), batch_size):
        batch = manifest[batch_start:batch_start + batch_size]

        # Prepare argument tuples for starmap
        batch_args = [(item, project_id, client, pbar, pbar_lock) for item in batch]

        pool = ThreadPool(n_processes) #number of workers
        new_subjects = pool.starmap(save_subject_wrapper, batch_args) #uploads a batch of subjects (each with two images) 
        pool.close()
        pool.join()
        
        
        # don't do this --- ??? why not?
        with client:
            try:
                subject_set.add(new_subjects) #link the newly uploaded batch of subjects to the existing subject set
            except:
                subject_set.reload()
                subject_set.add(new_subjects)

        # Now that they're added, update the txt file for each id
        for item in batch:
            update_txt_file(item['id_str'])

    pbar.close()
    print("All uploads completed.")

def save_subject_wrapper(manifest_item, gz_project_id, client, pbar=None, pbar_lock=None):
    """
    Create a single subject (two images) to be uploaded to Galaxy Zoo.

    Parameters:
    ----------
    manifest_item: dict
        A dictionary containing the ID, PNG location of images, and metadata to be uploaded
    gz_project_id: str
        Zooniverse project ID
    client: panoptes_client
        the client object used to connect locally to a remote hosted Zooniverse account.
        client = Panoptes.client(username=getpass.getpass('username: '), password=getpass.getpass('password: '))
    pbar: tqdm progress bar
        tqdm progress bar for tracking uploads. tqdm(total=len(manifest), unit=' subjects uploaded')
    pbar_lock: 
        keeps multiple threads from updated the progress bar at once (which could corrupt the display). threading.Lock()
    """
    subject = Subject()
    subject.links.project = gz_project_id

    for img_loc in manifest_item['png_locs']:
        assert os.path.exists(img_loc), f"Image not found: {img_loc}"
        subject.add_location(img_loc)

    subject.metadata.update(manifest_item['metadata'])
    subject.save(client=client)

    if pbar and pbar_lock:
        with pbar_lock:
            pbar.update()

    return subject


## Creating Our Manifest File

Some helpful links: 
- [What is a manifest](https://help.zooniverse.org/getting-started/example/#details-subject-sets-and-manifest-details-aka-what-is-a-manifest)
- [FAQ: Metadata, manifests, & more](https://docs.google.com/document/d/16kAtOeFvYYUbxY_1Yi9lhFQ2lvhFNL8-w7khUe7T2fs/edit?tab=t.0)
- [Example manifest file for multi-image subjects](https://docs.google.com/spreadsheets/d/1t8vlOpryNWksWuY-2XToB6EEa_CmnZxh2hHTXzp6GeM/edit?gid=0#gid=0)


In [5]:
# load in source catalog to add metadata to subjects
bands = ['f106', 'f129', 'f158']

subject_png_save_dir = '/teams/gz-roman/10012026-test-subjects_pngs/' #where to save the individual subject (2 panel) images

catalog_name = '10012026-TEST-full_catalog_file_locs_'
for i in range(len(bands)):
    if bands[i] == bands[-1]:
        catalog_name = catalog_name + bands[i]
    else:
        catalog_name = catalog_name + bands[i]+'_'
print(catalog_name)

cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')
print(cat_df.columns)

num_subjects = 200 # how many subjects do you want to create?

cat_df = cat_df[:num_subjects]
print(len(cat_df))

10012026-TEST-full_catalog_file_locs_f106_f129_f158
Index(['id_str', 'asdf_loc_f106', 'asdf_loc_f129', 'asdf_loc_f158',
       'ra_centroid_f106', 'dec_centroid_f106', 'kron_abmag_f106',
       'kron_radius_f106', 'fluxfrac_radius_50_f106', 'segment_area_f106',
       'is_extended_f106', 'semimajor_f106', 'semiminor_f106',
       'ra_centroid_f129', 'dec_centroid_f129', 'kron_abmag_f129',
       'kron_radius_f129', 'fluxfrac_radius_50_f129', 'segment_area_f129',
       'is_extended_f129', 'semimajor_f129', 'semiminor_f129',
       'ra_centroid_f158', 'dec_centroid_f158', 'kron_abmag_f158',
       'kron_radius_f158', 'fluxfrac_radius_50_f158', 'segment_area_f158',
       'is_extended_f158', 'semimajor_f158', 'semiminor_f158', 'skycell'],
      dtype='str')
200


#### Create a catalog of asdf file locations, source catalog metadata (AB mags and kron sizes), PNG file names, and external links. This catalog is used to attach provide image locations for, and attach metadata to, subject uplaods.

In [6]:

master_cat_metadata = '/teams/gz-roman/10012026-TEST-colour_cutout_master_catalogue_w_metadata.csv' 

# don't overwrite the catalog if it already exists
if not os.path.isfile(master_cat_metadata):
    
    image1_paths = []
    image2_paths = []
    ras = []
    decs = []
    desi_links = []
    ned_links = []
    esasky_links = []
    
    # for i in range(len(cat_df)):
    for src_i in trange(num_subjects):
        id_str = cat_df['id_str'][src_i]
        image1_paths.append(str(id_str) + '-composite.png')
        image2_paths.append(str(id_str) + '-single_band.png')
    
        ra = np.round(cat_df.loc[id_str][f'ra_centroid_{bands[0]}'],8)
        dec = np.round(cat_df.loc[id_str][f'dec_centroid_{bands[0]}'],8)
    
        desi_link = '[Click to view in DESI](+tab+https://www.legacysurvey.org/viewer?ra='+str(ra)+'&dec='+str(dec)+'&layer=ls-dr9&zoom=15)'
        ned_link = '[Click to search NASA NED](+tab+https://ned.ipac.caltech.edu/conesearch?search_type=Near%20Position%20Search&in_csys=Equatorial&in_equinox=J2000&ra='+str(ra)+'d&dec='+str(dec)+'d&radius=0.1)'
        esasky_link = '[Click to view in ESA Sky](+tab+https://sky.esa.int/esasky/?target='+str(ra)+'%'+str(dec)+'&hips=JWST+NIRCam+Imaging&fov=0.3940858692337921&projection=TAN&cooframe=J2000&sci=false&lang=en)'
    
        ras.append(ra)
        decs.append(dec)
        desi_links.append(desi_link)
        ned_links.append(ned_link)
        esasky_links.append(esasky_link)
    
    cat_df['image1_path'] = image1_paths
    cat_df['image2_path'] = image2_paths
    cat_df['!RA'] = ras
    cat_df['!DEC'] = decs
    cat_df['!desi_link'] = desi_links
    cat_df['!ned_link'] = ned_links
    cat_df['!esasky_link'] = esasky_links
    
    cat_df['#id_str'] = cat_df['id_str']
    
    # # Add in the COSMOS fit info
    # cutout_data['!radius'] = cosmos_info.loc[cutout_data['id_str']]['RADIUS'].values
    # cutout_data['!sersic_index'] = cosmos_info.loc[cutout_data['id_str']]['SERSIC'].values
    # cutout_data['!F444W_magnitude'] = cutout_data['MAG_MODEL_F444W']
    
    # Add link to COSMOS info
    # cutout_data['COSMOS_info'] = 'Thanks to the COSMOS-Web team for their support! [Learn more about COSMOS-Web here.](+tab+https://cosmos.astro.caltech.edu/page/cosmosweb)'
    
    cat_df.to_csv(master_cat_metadata, index=False)

100%|██████████| 200/200 [00:00<00:00, 5058.22it/s]


In [7]:
cat_df = pd.read_csv(master_cat_metadata)

In [8]:
print(cat_df.columns)
# print(cat_df.iloc[:5])

Index(['id_str', 'asdf_loc_f106', 'asdf_loc_f129', 'asdf_loc_f158',
       'ra_centroid_f106', 'dec_centroid_f106', 'kron_abmag_f106',
       'kron_radius_f106', 'fluxfrac_radius_50_f106', 'segment_area_f106',
       'is_extended_f106', 'semimajor_f106', 'semiminor_f106',
       'ra_centroid_f129', 'dec_centroid_f129', 'kron_abmag_f129',
       'kron_radius_f129', 'fluxfrac_radius_50_f129', 'segment_area_f129',
       'is_extended_f129', 'semimajor_f129', 'semiminor_f129',
       'ra_centroid_f158', 'dec_centroid_f158', 'kron_abmag_f158',
       'kron_radius_f158', 'fluxfrac_radius_50_f158', 'segment_area_f158',
       'is_extended_f158', 'semimajor_f158', 'semiminor_f158', 'skycell',
       'image1_path', 'image2_path', '!RA', '!DEC', '!desi_link', '!ned_link',
       '!esasky_link', '#id_str'],
      dtype='str')


In [9]:
# the columns of the cutout data we want to upload in the manifest
manifest_columns = ['image1_path',
       'image2_path', '!RA', '!DEC', '!desi_link', '!ned_link', '!esasky_link',
       '#id_str']

In [10]:
Panoptes.connect(username=getpass.getpass('username: '), password=getpass.getpass('password: '))

username:  ········
password:  ········


In [11]:
# project ID
test_project_id = 5733 #GZ project id
test_num = 4
new_subject_set_name = f'Roman-simulated-HLWAS-sources-test{test_num}' #enter name for new subject set to be uploaded

# 34152 #project on my personal account

In [12]:
test_subject_set_id = None

project = Project.find(test_project_id)


subject_set_ids = project.links.subject_sets
subject_set_names = [SubjectSet.find(subject_set_ids[i].id).display_name for i in range(len(list(subject_set_ids)))]

new_sub_index = np.where(np.array(subject_set_names) == new_subject_set_name)[0]

if len(new_sub_index)>0:
    print(f"Subject set {new_subject_set_name} already exists with subject set id = {list(subject_set_ids)[new_sub_index[0]].id}!")
    subject_set = list(subject_set_ids)[new_sub_index[0]]
    test_subject_set_id = list(subject_set_ids)[new_sub_index[0]].id
else:
    print(f'Attempting to edit project: {project.title}')
    print(f'Creating new subject set titled: {new_subject_set_name}!')
    
    Subject_set = SubjectSet()
    Subject_set.links.project = project
    Subject_set.display_name = new_subject_set_name
    Subject_set.save()
    test_subject_set_id = Subject_set.id
    print(f'New subject id is: {test_subject_set_id}')



Attempting to edit project: Galaxy Zoo
Creating new subject set titled: Roman-simulated-HLWAS-sources-test4!
New subject id is: 139555


In [13]:
# For tracking which subjects have already been uploaded
# Create the file initially (if it doesn't exist)
filename = f'/teams/gz-roman/uploaded_subjects_by_id_field-test{test_num}.txt'

# Make sure the file exists
open(filename, 'a').close()

In [15]:
# Get total logical CPU threads available on the hardware
cpu_threads = os.cpu_count()
print(f"Hardware CPU threads available: {cpu_threads}")
use_threads = cpu_threads

Hardware CPU threads available: 8


In [16]:
# cat_df.loc[0]

In [17]:
upload_selected_galaxies(
    cat_df,
    project_id = test_project_id,
    subject_set_id=test_subject_set_id,  
    subject_png_save_dir=subject_png_save_dir,
    manifest_columns=manifest_columns,
    progress_file=filename,
    n_processes=use_threads,
    batch_size=100,
)

username:  ········
password:  ········


uploaded_ids []
entry {'png_locs': ['/teams/gz-roman/10012026-test-subjects_pngs/0-composite.png', '/teams/gz-roman/10012026-test-subjects_pngs/0-single_band.png'], 'metadata': {'image1_path': '0-composite.png', 'image2_path': '0-single_band.png', '!RA': '9.42707326', '!DEC': '-45.59696241', '!desi_link': '[Click to view in DESI](+tab+https://www.legacysurvey.org/viewer?ra=9.42707326&dec=-45.59696241&layer=ls-dr9&zoom=15)', '!ned_link': '[Click to search NASA NED](+tab+https://ned.ipac.caltech.edu/conesearch?search_type=Near%20Position%20Search&in_csys=Equatorial&in_equinox=J2000&ra=9.42707326d&dec=-45.59696241d&radius=0.1)', '!esasky_link': '[Click to view in ESA Sky](+tab+https://sky.esa.int/esasky/?target=9.42707326%-45.59696241&hips=JWST+NIRCam+Imaging&fov=0.3940858692337921&projection=TAN&cooframe=J2000&sci=false&lang=en)', '#id_str': '0'}, 'id_str': '0'}
entry {'png_locs': ['/teams/gz-roman/10012026-test-subjects_pngs/1-composite.png', '/teams/gz-roman/10012026-test-subjects_pngs

100%|██████████| 200/200 [00:15<00:00, 12.81 subjects uploaded/s]

All uploads completed.
